In [11]:
import pandas as pd
import numpy as np

import librosa
import librosa.display

from pathlib import Path

from scipy.stats import skew, kurtosis
from scipy.stats import entropy as shannon_entropy

import matplotlib.pyplot as plt

In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = (
    PROJECT_ROOT /
    "dataset" /
    "ESC-50-master" /
    "ESC-50-master"
)

META_PATH = DATASET_PATH / "meta" / "esc50.csv"

AUDIO_PATH = DATASET_PATH / "audio"

print(META_PATH.exists())
print(AUDIO_PATH.exists())

True
True


In [3]:
# Load metadata

df = pd.read_csv(META_PATH)

natural_categories = [
    "chirping_birds",
    "crow",
    "crickets",
    "frog",
    "insects",
    "rain",
    "wind",
    "thunderstorm",
    "sea_waves",
    "water_drops",
    "pouring_water",
    "crackling_fire",
    "rooster"
]

natural_df = df[
    df["category"].isin(natural_categories)
]

print(natural_df.shape)

(520, 7)


In [12]:
def extract_features(signal, sr):

    # ---------- Basic Statistics ----------
    mean_amp = np.mean(signal)
    std_amp = np.std(signal)
    min_amp = np.min(signal)
    max_amp = np.max(signal)

    skewness = skew(signal)
    kurt = kurtosis(signal)

    # ---------- RMS Energy ----------
    rms = np.sqrt(np.mean(signal**2))

    # ---------- Shannon Entropy ----------
    hist, _ = np.histogram(
        signal,
        bins=100,
        density=True
    )

    hist = hist / np.sum(hist)

    entropy_value = shannon_entropy(hist)

    # ---------- Zero Crossing Rate ----------
    zcr = np.mean(
        librosa.feature.zero_crossing_rate(signal)
    )

    # ---------- Spectral Centroid ----------
    spectral_centroid = np.mean(
        librosa.feature.spectral_centroid(
            y=signal,
            sr=sr
        )
    )

    # ---------- Spectral Bandwidth ----------
    spectral_bandwidth = np.mean(
        librosa.feature.spectral_bandwidth(
            y=signal,
            sr=sr
        )
    )

    # ---------- Spectral Rolloff ----------
    spectral_rolloff = np.mean(
        librosa.feature.spectral_rolloff(
            y=signal,
            sr=sr
        )
    )

    return {

        "mean": mean_amp,
        "std": std_amp,
        "min": min_amp,
        "max": max_amp,

        "skewness": skewness,
        "kurtosis": kurt,

        "rms": rms,

        "entropy": entropy_value,

        "zcr": zcr,

        "spectral_centroid": spectral_centroid,
        "spectral_bandwidth": spectral_bandwidth,
        "spectral_rolloff": spectral_rolloff
    }

In [5]:
sample_row = natural_df.iloc[0]

audio_file = AUDIO_PATH / sample_row["filename"]

signal, sr = librosa.load(audio_file, sr=None)

test_features = extract_features(signal, sr)

test_features

{'mean': np.float32(-1.3780659e-06),
 'std': np.float32(0.061433855),
 'min': np.float32(-0.48049927),
 'max': np.float32(0.54556274),
 'skewness': np.float32(0.3929884),
 'kurtosis': np.float32(10.404342),
 'rms': np.float32(0.061433855),
 'entropy': np.float64(-229.37693026843453),
 'zcr': np.float64(0.15799353792053364),
 'spectral_centroid': np.float64(4042.2812036202286),
 'spectral_bandwidth': np.float64(2643.1096962158636),
 'spectral_rolloff': np.float64(6623.4833825768565)}

In [13]:
all_features = []

for _, row in natural_df.iterrows():

    audio_file = AUDIO_PATH / row["filename"]

    signal, sr = librosa.load(
        audio_file,
        sr=None
    )

    feature_dict = extract_features(
        signal,
        sr
    )

    feature_dict["filename"] = row["filename"]
    feature_dict["category"] = row["category"]

    all_features.append(feature_dict)

features_df = pd.DataFrame(all_features)

print(features_df.shape)

features_df.head()

(520, 14)


,mean,std,min,max,skewness,kurtosis,rms,entropy,zcr,spectral_centroid,spectral_bandwidth,spectral_rolloff,filename,category
0,-0.000001,0.061434,-0.480499,0.545563,0.392988,10.404342,0.061434,2.948086,0.157994,4042.281204,2643.109696,6623.483383,1-100038-A-14.wav,chirping_birds
1,-0.000003,0.010946,-0.082306,0.102997,0.157899,8.572530,0.010946,3.013087,0.027574,2075.738865,3455.603120,4559.843116,1-101296-A-19.wav,thunderstorm
2,-0.000010,0.016860,-0.164459,0.156952,-0.039740,10.883121,0.016860,2.866590,0.028877,1850.646524,3093.555844,3547.033153,1-101296-B-19.wav,thunderstorm
3,-0.000253,0.136177,-0.945587,0.999969,0.027326,2.776779,0.136177,3.313940,0.041329,2675.116865,3530.183905,6261.615656,1-103298-A-9.wav,crow
4,-0.000025,0.063967,-0.816681,0.980133,-0.052734,29.311878,0.063967,2.170481,0.010723,446.951318,1005.672928,696.156966,1-115521-A-19.wav,thunderstorm


In [14]:
features_df.to_csv(
    PROJECT_ROOT / "outputs" / "natural_sound_features.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [15]:
from scipy.stats import entropy

In [10]:
sample_entropy = shannon_entropy(signal)

print(sample_entropy)

2.5221894751282976


In [16]:
features_df["entropy"].describe()

count    520.000000
mean       2.705459
std        0.956360
min        0.073366
25%        2.031649
50%        2.943688
75%        3.459628
max        4.548030
Name: entropy, dtype: float64

In [17]:
features_df["entropy"].head()

0    2.948086
1    3.013087
2    2.866590
3    3.313940
4    2.170481
Name: entropy, dtype: float64

In [18]:
features_df.to_csv(
    PROJECT_ROOT / "outputs" / "natural_sound_features.csv",
    index=False
)

print("Version 1.0 Saved")

Version 1.0 Saved
